# Subtask 4 (part 1): Zero-shot module composition + baselines

Evaluates the two independently-trained LoRA modules (`models/genre_module_dgt`,
`models/language_module_books_fi`) on the target-domain test set
(`data/target_domain_fi_dgt/test.jsonl`, Finnish DGT), without any additional training.

## Conditions

| name | adapters active | purpose |
|---|---|---|
| `backbone` | none | floor baseline: plain `xglm-564M` |
| `genre_only` | genre | register-adapted, but never saw Finnish |
| `language_only` | language | Finnish-adapted, but never saw legal/admin register |
| `composed` | genre + language | zero-shot composition: does putting both together help? |


## 0. Setup

In [ ]:
import gc
import time
from pathlib import Path

import torch
import torch.nn as nn
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling

SEED = 42
DEVICE = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

MODEL_NAME = "facebook/xglm-564M"   # same backbone used to train both LoRA modules
DATA_DIR = Path("data")
MODELS_DIR = Path("models")

# the two independently-trained adapters 
GENRE_ADAPTER = MODELS_DIR / "genre_module_dgt"
LANGUAGE_ADAPTER = MODELS_DIR / "language_module_books_fi"

MAX_LENGTH = 128     # same truncation used to train both modules, kept consistent for fair comparison
EVAL_BATCH_SIZE = 16 # inference only (no gradients), so this can be larger than the training batch size

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device: mps


## 1. Target-domain test set

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # XGLM has no dedicated pad token

def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH)

test_ds = load_dataset("json", data_files=str(DATA_DIR / "target_domain_fi_dgt" / "test.jsonl"), split="train")
test_ds = test_ds.map(tokenize, remove_columns=test_ds.column_names)
print(f"test examples: {len(test_ds):,}")


collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
# shuffle=False is important: keeps sentence order fixed across every condition/notebook so per-example
# loss arrays line up index-for-index later, which is what a paired significance test needs
test_loader = DataLoader(test_ds, batch_size=EVAL_BATCH_SIZE, shuffle=False, collate_fn=collator)

test examples: 2,618


## 2. Model-loading helpers


In [3]:
def load_condition(adapters=None, weights=None, combination_type="linear"):
    """adapters: None -> plain backbone (no adapters at all).
    A list of (name, path) pairs -> that/those adapter(s) active.
    Two entries -> combined into one adapter via PEFT's add_weighted_adapter before use.
    """
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.bfloat16)

    if not adapters:
        # baseline condition: no PEFT wrapper at all, just the frozen pretrained backbone
        base.to(DEVICE)
        base.eval()
        return base

    names = [n for n, _ in adapters]
    first_name, first_path = adapters[0]
    # first adapter is loaded via from_pretrained (wraps the base model in a PeftModel);
    # any additional adapters are loaded into that same PeftModel via load_adapter
    model = PeftModel.from_pretrained(base, first_path, adapter_name=first_name)
    for name, path in adapters[1:]:
        model.load_adapter(str(path), adapter_name=name)

    if len(adapters) == 1:
        # single-module condition (genre_only / language_only): just activate that one adapter
        model.set_adapter(first_name)
    else:
        # composition step: weighted sum of the LoRA deltas in weight space,
        # ΔW_composed = w_genre * ΔW_genre + w_language * ΔW_language (equal weights by default)
        w = weights if weights is not None else [1.0] * len(adapters)
        model.add_weighted_adapter(
            adapters=names, weights=w, adapter_name="composed", combination_type=combination_type,
        )
        model.set_adapter("composed")

    model.to(DEVICE)
    model.eval()
    return model


def free(*objs):
    """Explicit cleanup between conditions -- each load_condition() call loads a fresh copy of the
    564M-param backbone, so freeing the previous one before loading the next keeps memory bounded."""
    for o in objs:
        del o
    gc.collect()
    if DEVICE == "mps":
        torch.mps.empty_cache()
    elif DEVICE == "cuda":
        torch.cuda.empty_cache()

## 3. Evaluation helper (per-sequence NLL)

In [ ]:
@torch.no_grad()
def evaluate(model, loader, label=""):
    """Per-sequence average negative log-likelihood (natural log), computed manually rather than via
    Trainer.evaluate() so to keep a per-EXAMPLE array (needed later for a paired significance test),
    not just a single corpus-level mean.

    IMPORTANT: 'labels' is deliberately kept OUT of the model(**inputs) call. XGLM has a ~256k-token
    vocabulary, so if 'labels' is passed, HF computes its own internal full-vocab cross-entropy loss
    on top of the one that is computed manually below. 
    """
    # Reduction='none' > returns one loss value per token
    loss_fct = nn.CrossEntropyLoss(reduction="none", ignore_index=-100) # Asking: how much probability did the model assign to the token that was actually correct?
    per_example_losses = []

    t0 = time.time()
    for i, batch in enumerate(loader):
        model_inputs = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
        labels = batch["labels"].to(DEVICE)
        logits = model(**model_inputs).logits

        # standard causal-LM shift: position t's logits predict token t+1
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()


        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1)
        ).view(shift_labels.size())

        # mask out padded positions (label == -100) before averaging within each sequence
        mask = (shift_labels != -100).float()
        seq_losses = (token_losses * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        per_example_losses.extend(seq_losses.float().cpu().tolist())

        # MPS does not eagerly release freed memory between iterations -- without periodic
        # empty_cache() calls, per-batch latency creeps up noticeably over a ~164-batch sweep
        if (i + 1) % 4 == 0 and DEVICE == "mps":
            torch.mps.empty_cache()
        if (i + 1) % 20 == 0:
            print(f"  [{label}] batch {i+1}/{len(loader)} ({time.time()-t0:.0f}s elapsed)", flush=True)

    per_example_losses = torch.tensor(per_example_losses)
    mean_loss = per_example_losses.mean().item()
    print(f"  [{label}] done in {time.time()-t0:.0f}s")
    return {
        "mean_loss": mean_loss,
        "ppl": float(torch.exp(torch.tensor(mean_loss))),  # perplexity = exp(mean NLL)
        "per_example_losses": per_example_losses,           # kept for the later paired stats test
    }

## 4. Run the four zero-shot conditions


### 4a. `backbone` (no adapters)

In [5]:
results = {}

# condition 1/4: floor baseline -- plain backbone, no adaptation of any kind
model = load_condition(None)
results["backbone"] = evaluate(model, test_loader, label="backbone")
free(model)
print(f"mean_loss={results['backbone']['mean_loss']:.4f}  ppl={results['backbone']['ppl']:.2f}")

  [backbone] batch 20/164 (29s elapsed)


  [backbone] batch 40/164 (62s elapsed)


  [backbone] batch 60/164 (95s elapsed)


  [backbone] batch 80/164 (132s elapsed)


  [backbone] batch 100/164 (166s elapsed)


  [backbone] batch 120/164 (203s elapsed)


  [backbone] batch 140/164 (254s elapsed)


  [backbone] batch 160/164 (303s elapsed)


  [backbone] done in 311s


mean_loss=3.5184  ppl=33.73


### 4b. `genre_only`

In [ ]:
# condition 2/4: genre module only -- register-adapted, but this module never saw Finnish
# (trained on OPUS DGT in every language EXCEPT Finnish)
model = load_condition([("genre", GENRE_ADAPTER)])
results["genre_only"] = evaluate(model, test_loader, label="genre_only")
free(model)
print(f"mean_loss={results['genre_only']['mean_loss']:.4f}  ppl={results['genre_only']['ppl']:.2f}")

  [genre_only] batch 20/164 (38s elapsed)


  [genre_only] batch 40/164 (80s elapsed)


  [genre_only] batch 60/164 (125s elapsed)


  [genre_only] batch 80/164 (171s elapsed)


  [genre_only] batch 100/164 (215s elapsed)


  [genre_only] batch 120/164 (263s elapsed)


  [genre_only] batch 140/164 (308s elapsed)


  [genre_only] batch 160/164 (347s elapsed)


  [genre_only] done in 354s


mean_loss=3.4674  ppl=32.05


### 4c. `language_only`

In [ ]:
# condition 3/4: language module only -- Finnish-adapted, but this module never saw legal/admin
# register (trained on the Finnish portion of OPUS Books, i.e. literary text)
model = load_condition([("language", LANGUAGE_ADAPTER)])
results["language_only"] = evaluate(model, test_loader, label="language_only")
free(model)
print(f"mean_loss={results['language_only']['mean_loss']:.4f}  ppl={results['language_only']['ppl']:.2f}")

  [language_only] batch 20/164 (39s elapsed)


  [language_only] batch 40/164 (81s elapsed)


  [language_only] batch 60/164 (122s elapsed)


  [language_only] batch 80/164 (171s elapsed)


  [language_only] batch 100/164 (217s elapsed)


  [language_only] batch 120/164 (262s elapsed)


  [language_only] batch 140/164 (305s elapsed)


  [language_only] batch 160/164 (343s elapsed)


  [language_only] done in 350s


mean_loss=3.7689  ppl=43.33


### 4d. `composed` (genre + language, weighted sum)

In [8]:
# condition 4/4: both modules composed via weighted sum (equal weights) -- the actual
# zero-shot composition test this subtask is asking about
model = load_condition([("genre", GENRE_ADAPTER), ("language", LANGUAGE_ADAPTER)])
results["composed"] = evaluate(model, test_loader, label="composed")
free(model)
print(f"mean_loss={results['composed']['mean_loss']:.4f}  ppl={results['composed']['ppl']:.2f}")

  [composed] batch 20/164 (39s elapsed)


  [composed] batch 40/164 (920s elapsed)


  [composed] batch 60/164 (958s elapsed)


  [composed] batch 80/164 (997s elapsed)


  [composed] batch 100/164 (1034s elapsed)


  [composed] batch 120/164 (1071s elapsed)


  [composed] batch 140/164 (1110s elapsed)


  [composed] batch 160/164 (1143s elapsed)


  [composed] done in 1149s


mean_loss=3.8603  ppl=47.48


## 5. Order sanity check

Composing `(genre, language)` vs `(language, genre)` with the linear/additive operator should give
numerically identical per-example losses, since weight-space addition is commutative.

In [9]:
# order check, pass 1/2: genre listed first, language second.
# Note this is really just a label for add_weighted_adapter's `adapters=` argument order --
# since combination_type="linear" is a plain weighted SUM, it is commutative by construction, so
# this and the next cell are expected to produce numerically identical results (verified below).
model_gl = load_condition([("genre", GENRE_ADAPTER), ("language", LANGUAGE_ADAPTER)])
out_gl = evaluate(model_gl, test_loader, label="genre-then-language")
free(model_gl)

  [genre-then-language] batch 20/164 (31s elapsed)


  [genre-then-language] batch 40/164 (65s elapsed)


  [genre-then-language] batch 60/164 (100s elapsed)


  [genre-then-language] batch 80/164 (526s elapsed)


  [genre-then-language] batch 100/164 (565s elapsed)


  [genre-then-language] batch 120/164 (604s elapsed)


  [genre-then-language] batch 140/164 (642s elapsed)


  [genre-then-language] batch 160/164 (676s elapsed)


  [genre-then-language] done in 683s


In [ ]:
# order check, pass 2/2: language listed first, genre second
model_lg = load_condition([("language", LANGUAGE_ADAPTER), ("genre", GENRE_ADAPTER)])
out_lg = evaluate(model_lg, test_loader, label="language-then-genre")
free(model_lg)

# To confirm that weight-space addition is order-invariant
max_abs_diff = (out_gl["per_example_losses"] - out_lg["per_example_losses"]).abs().max().item()
print(f"genre-then-language mean_loss={out_gl['mean_loss']:.6f}")
print(f"language-then-genre mean_loss={out_lg['mean_loss']:.6f}")
print(f"max per-example |diff| = {max_abs_diff:.2e}  (floating-point noise, not a real effect, if ~1e-5 or smaller)")

  [language-then-genre] batch 20/164 (34s elapsed)


  [language-then-genre] batch 40/164 (71s elapsed)


  [language-then-genre] batch 60/164 (108s elapsed)


  [language-then-genre] batch 80/164 (168s elapsed)


  [language-then-genre] batch 100/164 (214s elapsed)


  [language-then-genre] batch 120/164 (259s elapsed)


  [language-then-genre] batch 140/164 (303s elapsed)


  [language-then-genre] batch 160/164 (343s elapsed)


  [language-then-genre] done in 351s


genre-then-language mean_loss=3.860293
language-then-genre mean_loss=3.860293
max per-example |diff| = 0.00e+00  (floating-point noise, not a real effect, if ~1e-5 or smaller)


## 6. Results summary

In [11]:
import pandas as pd

summary = pd.DataFrame([
    {"condition": name, "mean_loss": out["mean_loss"], "ppl": out["ppl"]}
    for name, out in results.items()
])
summary

,condition,mean_loss,ppl
0,backbone,3.518378,33.729660
1,genre_only,3.467448,32.054825
2,language_only,3.768940,43.334106
3,composed,3.860293,47.479279


In [ ]:
import numpy as np

# per-example loss arrays
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
np.savez(
    RESULTS_DIR / "zero_shot_per_example_losses.npz",
    **{name: out["per_example_losses"].numpy() for name, out in results.items()},
)
print("saved per-example losses to results/zero_shot_per_example_losses.npz")

saved per-example losses to results/zero_shot_per_example_losses.npz
